# Exploratory Data Analysis - Satellite Change Detection Dataset

This notebook explores the structure of the satellite change detection dataset, analyzes class distributions, and visualizes typical change patterns.

## What we're investigating:
- Dataset structure and file alignment across train/val/test splits
- Visual examples of pre-change (T1), post-change (T2) image pairs and associated change masks
- Class imbalance characteristics (changed vs. unchanged pixels)
- Spectral properties and pixel intensity distributions
- Spatial characteristics of change regions (size, connectivity, prevalence)

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Configure data root - update this to your local path
DATA_ROOT = Path('../data/raw')
SPLITS = ['train', 'val', 'test']

print(f'Data root: {DATA_ROOT}')
print(f'Exists: {DATA_ROOT.exists()}')

## Dataset Overview

Count images per split and verify that A/, B/, and label/ directories are aligned.
- **A/**: Pre-change satellite images (T1)
- **B/**: Post-change satellite images (T2)
- **label/**: Binary change masks (0 = no change, 255 = change)

In [ ]:
def dataset_summary(data_root: Path) -> dict:
    """
    Count images per split and check directory alignment.
    
    Verifies that A/, B/, and label/ subdirectories exist and contain
    the same number of files for each split.
    
    Args:
        data_root: Root path to the dataset
        
    Returns:
        Dictionary mapping split name to file counts per subdirectory
    """
    summary = {}
    for split in SPLITS:
        split_dir = data_root / split
        if not split_dir.exists():
            print(f'  {split}: NOT FOUND')
            continue
        
        dirs = {'A': 'pre-change', 'B': 'post-change', 'label': 'change masks'}
        counts = {}
        for subdir, desc in dirs.items():
            path = split_dir / subdir
            if path.exists():
                n = len(list(path.glob('*')))
                counts[subdir] = n
                print(f'  {split}/{subdir} ({desc}): {n} files')
            else:
                print(f'  {split}/{subdir}: MISSING')
        
        # Check alignment
        unique_counts = set(counts.values())
        if len(unique_counts) == 1:
            print(f'  -> All aligned ({unique_counts.pop()} files per directory)')
        else:
            print(f'  -> WARNING: Misaligned directories!')
        summary[split] = counts
        print()
    return summary

summary = dataset_summary(DATA_ROOT)

## Visualize Image Pairs

Display pre-change (T1), post-change (T2), and change mask side by side.
This helps us understand what kind of changes the dataset captures:
- Urban development (new buildings, roads)
- Vegetation loss (deforestation, agricultural changes)
- Water body changes
- Disaster impact assessment

In [ ]:
def show_samples(data_root: Path, split: str = 'train', n_samples: int = 4):
    """
    Display a grid of image pair samples with their change masks.
    
    Shows pre-change, post-change, and annotated change mask for each sample,
    allowing visual inspection of dataset characteristics and data quality.
    
    Args:
        data_root: Root path to the dataset
        split: Which split to sample from ('train', 'val', 'test')
        n_samples: Number of triplets to display
    """
    split_dir = data_root / split
    filenames = sorted(os.listdir(split_dir / 'A'))[:n_samples]
    
    fig, axes = plt.subplots(n_samples, 3, figsize=(12, 4 * n_samples))
    if n_samples == 1:
        axes = axes.reshape(1, -1)
    
    for i, fname in enumerate(filenames):
        # Load T1 (pre-change), T2 (post-change), and change mask
        img_a = np.array(Image.open(split_dir / 'A' / fname))
        img_b = np.array(Image.open(split_dir / 'B' / fname))
        mask = np.array(Image.open(split_dir / 'label' / fname).convert('L'))
        
        axes[i, 0].imshow(img_a)
        axes[i, 0].set_title(f'Pre-Change (T1)' if i == 0 else '')
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(img_b)
        axes[i, 1].set_title(f'Post-Change (T2)' if i == 0 else '')
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(mask, cmap='hot')
        axes[i, 2].set_title(f'Change Mask' if i == 0 else '')
        axes[i, 2].axis('off')
        
        axes[i, 0].set_ylabel(fname, fontsize=8)
    
    plt.suptitle(f'Sample Pairs from {split} split', fontsize=14)
    plt.tight_layout()
    plt.show()

# Uncomment when data is available:
# show_samples(DATA_ROOT, 'train', n_samples=4)

## Class Distribution Analysis

Change detection datasets are heavily imbalanced. Typically 90-99% of pixels
are 'no change'. Understanding this distribution helps us:
- Choose the right loss function (Dice loss, Focal loss, weighted BCE)
- Implement proper sampling strategies (hard negative mining, change-aware patching)
- Set appropriate class weights during training

In [ ]:
def analyze_class_distribution(data_root: Path, split: str = 'train'):
    """
    Compute per-image change fractions and plot distribution.
    
    Analyzes the fraction of changed pixels in each image to characterize
    the class imbalance and understand dataset bias.
    
    Args:
        data_root: Root path to the dataset
        split: Which split to analyze
    """
    label_dir = data_root / split / 'label'
    if not label_dir.exists():
        print(f'Label directory not found: {label_dir}')
        return
    
    fractions = []
    for fname in sorted(os.listdir(label_dir)):
        mask = np.array(Image.open(label_dir / fname).convert('L'))
        # Binarize: some datasets use 0/255, others use 0/1
        binary_mask = (mask > 128).astype(float)
        change_fraction = binary_mask.mean()
        fractions.append(change_fraction)
    
    fractions = np.array(fractions)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram of change fractions
    ax1.hist(fractions, bins=50, color='steelblue', edgecolor='white')
    ax1.axvline(fractions.mean(), color='red', linestyle='--', label=f'Mean: {fractions.mean():.4f}')
    ax1.set_xlabel('Fraction of Changed Pixels')
    ax1.set_ylabel('Number of Images')
    ax1.set_title(f'Change Distribution ({split})')
    ax1.legend()
    
    # Cumulative distribution
    sorted_fracs = np.sort(fractions)
    ax2.plot(sorted_fracs, np.linspace(0, 1, len(sorted_fracs)))
    ax2.set_xlabel('Change Fraction')
    ax2.set_ylabel('Cumulative Proportion')
    ax2.set_title('CDF of Change Fractions')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print(f'Total images: {len(fractions)}')
    print(f'Images with any change: {(fractions > 0.001).sum()} ({(fractions > 0.001).mean()*100:.1f}%)')
    print(f'Mean change fraction: {fractions.mean():.4f}')
    print(f'Median change fraction: {np.median(fractions):.4f}')
    print(f'Max change fraction: {fractions.max():.4f}')

# Uncomment when data is available:
# analyze_class_distribution(DATA_ROOT, 'train')

## Spectral Analysis

For multispectral data (like Sentinel-2), analyzing per-band statistics helps us:
- Decide on normalization strategy
- Identify any band-specific issues (saturation, noise)
- Determine if band-specific preprocessing is needed

In [ ]:
def analyze_pixel_distributions(data_root: Path, split: str = 'train', max_images: int = 100):
    """
    Plot pixel intensity distributions for each channel.
    
    Computes mean and standard deviation per channel across the dataset
    to inform normalization strategies.
    
    Args:
        data_root: Root path to the dataset
        split: Which split to analyze
        max_images: Maximum number of images to process
    """
    img_dir = data_root / split / 'A'
    if not img_dir.exists():
        print(f'Image directory not found: {img_dir}')
        return
    
    filenames = sorted(os.listdir(img_dir))[:max_images]
    
    # Collect per-channel statistics
    all_means = []
    all_stds = []
    
    for fname in filenames:
        img = np.array(Image.open(img_dir / fname)).astype(float)
        if img.ndim == 2:
            img = img[:, :, np.newaxis]
        for c in range(img.shape[2]):
            while len(all_means) <= c:
                all_means.append([])
                all_stds.append([])
            all_means[c].append(img[:, :, c].mean())
            all_stds[c].append(img[:, :, c].std())
    
    n_channels = len(all_means)
    channel_names = ['Red', 'Green', 'Blue'][:n_channels] if n_channels <= 3 else [f'Band {i}' for i in range(n_channels)]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Mean per channel
    means = [np.mean(m) for m in all_means]
    stds = [np.mean(s) for s in all_stds]
    x = range(n_channels)
    
    ax1.bar(x, means, color=['#e74c3c', '#2ecc71', '#3498db'][:n_channels])
    ax1.set_xticks(x)
    ax1.set_xticklabels(channel_names)
    ax1.set_ylabel('Mean Pixel Value')
    ax1.set_title('Mean Pixel Value per Channel')
    
    ax2.bar(x, stds, color=['#e74c3c', '#2ecc71', '#3498db'][:n_channels])
    ax2.set_xticks(x)
    ax2.set_xticklabels(channel_names)
    ax2.set_ylabel('Std Pixel Value')
    ax2.set_title('Pixel Value Std per Channel')
    
    plt.tight_layout()
    plt.show()
    
    print('Per-channel statistics (computed over training set):')
    for i, name in enumerate(channel_names):
        print(f'  {name}: mean={means[i]:.2f}, std={stds[i]:.2f}')

# Uncomment when data is available:
# analyze_pixel_distributions(DATA_ROOT, 'train')

## Change Mask Statistics

Analyzing the size and shape of change regions helps us:
- Choose appropriate patch sizes (larger patches for spatially dispersed changes)
- Understand what spatial resolution the model needs
- Design preprocessing strategies (morphological operations, small-region filtering)

In [ ]:
from scipy import ndimage

def analyze_change_regions(data_root: Path, split: str = 'train', max_images: int = 200):
    """
    Analyze connected components in change masks.
    
    Identifies individual change regions and computes size statistics
    to understand the spatial characteristics of changes.
    
    Args:
        data_root: Root path to the dataset
        split: Which split to analyze
        max_images: Maximum number of images to process
    """
    label_dir = data_root / split / 'label'
    if not label_dir.exists():
        print(f'Label directory not found: {label_dir}')
        return
    
    region_sizes = []
    region_counts = []
    
    for fname in sorted(os.listdir(label_dir))[:max_images]:
        mask = np.array(Image.open(label_dir / fname).convert('L'))
        binary = (mask > 128).astype(int)
        
        # Find connected components (individual change regions)
        labeled, n_regions = ndimage.label(binary)
        region_counts.append(n_regions)
        
        for region_id in range(1, n_regions + 1):
            size = (labeled == region_id).sum()
            region_sizes.append(size)
    
    if not region_sizes:
        print('No change regions found.')
        return
    
    region_sizes = np.array(region_sizes)
    region_counts = np.array(region_counts)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Region size distribution (log scale due to heavy tail)
    ax1.hist(np.log10(region_sizes + 1), bins=50, color='coral', edgecolor='white')
    ax1.set_xlabel('log10(Region Size in Pixels)')
    ax1.set_ylabel('Count')
    ax1.set_title('Change Region Size Distribution')
    
    # Number of regions per image
    ax2.hist(region_counts, bins=30, color='mediumseagreen', edgecolor='white')
    ax2.set_xlabel('Number of Change Regions')
    ax2.set_ylabel('Number of Images')
    ax2.set_title('Change Regions per Image')
    
    plt.tight_layout()
    plt.show()
    
    print(f'Total change regions: {len(region_sizes)}')
    print(f'Median region size: {np.median(region_sizes):.0f} pixels')
    print(f'Mean regions per image: {region_counts.mean():.1f}')

# Uncomment when data is available:
# analyze_change_regions(DATA_ROOT, 'train')

## Key Observations

After running this analysis, document your findings here:

### 1. Class Imbalance
Typical change detection datasets have <5% changed pixels. This imbalance justifies:
- Using **Dice loss** or **Focal loss** instead of standard cross-entropy
- Implementing **oversampling** of change-containing patches during training
- Using **class weights** to penalize false negatives more heavily
- Applying **hard negative mining** to focus on difficult samples

### 2. Change Region Sizes
Most change regions are small (individual buildings, tree clusters). Implications:
- 256x256 patch size should capture most small changes
- Larger coordinated changes (subdivisions, deforestation tracts) may span multiple patches
- Consider multi-scale architectures (U-Net with skip connections) for handling different change scales

### 3. Spectral Properties
For normalization strategy:
- If using **pretrained ImageNet weights**: stick with ImageNet normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
- For **training from scratch** on multispectral data: compute dataset-specific statistics and apply z-score normalization
- Check for band-specific saturation or clipping artifacts

### 4. Data Quality Checks
Investigate potential issues:
- Cloud cover and shadows creating false changes
- Seasonal differences (vegetation phenology) if images are from different seasons
- Co-registration errors creating spurious edge artifacts
- Radiometric differences requiring histogram matching or other preprocessing